In [ ]:

import timeit


geom_results = {}

geom_results['Circle'] = benchmark(lambda: fwd_circle_central(
    thicknesses, resistivities, a, times,
    use_numba=False, use_cuda=False, hankel_filter='key_201'))

geom_results['Square\nFull'] = benchmark(lambda: fwd_square_central(
    thicknesses, resistivities, L, times,
    n_quad=5, use_symmetry=False, use_numba=False, use_cuda=False, hankel_filter='key_201'))

geom_results['Square\nSym'] = benchmark(lambda: fwd_square_central(
    thicknesses, resistivities, L, times,
    n_quad=5, use_symmetry=True, use_numba=False, use_cuda=False, hankel_filter='key_201'))

geom_results['Circle\n(offset)'] = benchmark(lambda: fwd_circle_offset(
    thicknesses, resistivities, a, rx_offset, times,
    use_numba=False, use_cuda=False, hankel_filter='key_201'))

geom_results['Square\n(offset)'] = benchmark(lambda: fwd_square_offset(
    thicknesses, resistivities, L, rx_offset, 0.0, times,
    n_quad=11, use_numba=False, use_cuda=False, hankel_filter='key_201'))

print(f"{'Geometry':25s}  {'Time [ms]':>10s}")
print("-" * 38)
for k, v in geom_results.items():
    print(f"  {k.replace(chr(10), ' '):23s}  {v * 1000:10.1f}")

# --- Bar chart ---
labels = [k.replace('\n', ' ') for k in geom_results]
vals   = [v for v in geom_results.values()]   # s

fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.bar(labels, vals, color='k', edgecolor='black', width=0.55)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width() / 2, val * 1.05,
            f"{val*1000:.0f} ms", ha='center', va='bottom', fontsize=8,
            bbox=dict(facecolor='white', edgecolor='k', alpha=1))
ax.set_ylabel('Runtime [s]')
ax.set_yscale('log')
ax.set_ylim(None, max(vals) * 2)
fig.tight_layout()
plt.show()


### 1. Baseline: single forward-call runtime

Before optimising, let's measure how long a single forward model call takes for each of the four loop geometries implemented pyTEM. The model used throughout this notebook has 30 layers and 31 gate times, representative of a real survey.

| Geometry | Loop type | Rx position |
|----------|-----------|-------------|
| Circle-central | Circular Tx loop | Rx at loop centre |
| Circle-offset | Circular Tx loop | Rx offset from centre |
| Square-central | Square Tx loop | Rx at loop centre |
| Square-offset | Square Tx loop | Rx offset from centre |

The bar chart below shows median forward-call time per geometry. Even the fastest call costs several milliseconds; consequently a full inversion runs into tens of seconds — motivating the optimisations in the sections below.